In [1]:
# ============================================================
# GLOBAL OEM PIT DATABASE — COMPACT DIAGNOSTIC PACKAGE
# ============================================================
#
# PURPOSE
# -------
# Diagnose the existing Blocks 1–10 WITHOUT modifying them.
#
# Main questions:
#   1. Which securities/issuers are likely OEMs?
#   2. Which canonical fields are actually present upstream?
#   3. Which fields disappear between regional blocks and Block 10?
#   4. Are missing fiscal fields actually derivable upstream?
#   5. Are security_id / ticker propagation problems occurring?
#   6. How much regional OEM coverage survives into Block 10?
#   7. Is Block 9 feedback visibly reaching Block 10?
#
# OUTPUT
# ------
# One Excel workbook:
#   OEM_Diagnostic_Package.xlsx
#
# Plus:
#   OEM_Diagnostic_Package.zip
#
# The ZIP contains the same diagnostic tables as CSV plus metadata JSON.
#
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import os
import sys
import zipfile
import warnings
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. GOOGLE DRIVE
# ------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped (not running in Colab or already mounted).")


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
)

DATA_ROOT = PROJECT_ROOT / "data" / "interim"

OUTPUT_DIR = PROJECT_ROOT / "data" / "diagnostics" / "oem_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = OUTPUT_DIR / "OEM_Diagnostic_Package.xlsx"
ZIP_PATH   = OUTPUT_DIR / "OEM_Diagnostic_Package.zip"
CSV_DIR    = OUTPUT_DIR / "csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "BLOCK1_SECURITY_SEED":
        DATA_ROOT / "block_1" / "security_master_seed_df.parquet",

    "BLOCK2_SECURITY_MASTER":
        DATA_ROOT / "block_2" / "security_master_df.parquet",

    "USA":
        DATA_ROOT / "block_3" / "sec_fundamentals_security_linked_df.parquet",

    "EUROPE":
        DATA_ROOT / "block_4" / "europe_fundamentals_security_linked_df.parquet",

    "JAPAN":
        DATA_ROOT / "block_5" / "japan_fundamentals_security_linked_df.parquet",

    "KOREA":
        DATA_ROOT / "block_6" / "korea_fundamentals_security_linked_df.parquet",

    "MAINLAND_CHINA":
        DATA_ROOT / "block_7" / "china_fundamentals_standardised_df.parquet",

    "HONG_KONG":
        DATA_ROOT / "block_7" / "hong_kong_fundamentals_standardised_df.parquet",

    "AUSTRALIA":
        DATA_ROOT / "block_8" / "australia_fundamentals_standardised_df.parquet",

    "BLOCK9":
        DATA_ROOT / "block_9" / "ai_exception_deduplicated_outcomes.parquet",

    "BLOCK10":
        DATA_ROOT / "block_10" / "global_fundamentals_selected_df.parquet",
}


# ------------------------------------------------------------
# 2. OEM CANDIDATE CLASSIFICATION
# ------------------------------------------------------------
#
# Conservative name matching.
# We can amend this after inspecting the candidate sheet.
#
# IMPORTANT:
# This does NOT alter the security master.
# It only creates a diagnostic OEM subset.
# ------------------------------------------------------------

OEM_PATTERNS = [
    # USA
    r"\btesla\b",
    r"\bgeneral motors\b",
    r"\bford motor\b",
    r"\brivian\b",
    r"\blucid\b",

    # Europe
    r"\bvolkswagen\b",
    r"\bbayerische motoren werke\b",
    r"\bbmw\b",
    r"\bmercedes[- ]benz\b",
    r"\bmercedes benz group\b",
    r"\bdaimler\b",
    r"\bstellantis\b",
    r"\brenault\b",
    r"\bferrari\b",
    r"\bporsche\b",
    r"\bvolvo car\b",
    r"\baston martin\b",

    # Japan
    r"\btoyota motor\b",
    r"\bhonda motor\b",
    r"\bnissan motor\b",
    r"\bmazda motor\b",
    r"\bsubaru\b",
    r"\bsuzuki motor\b",
    r"\bmitsubishi motors\b",
    r"\bisuzu motors\b",
    r"\bhino motors\b",

    # Korea
    r"\bhyundai motor\b",
    r"\bkia\b",
    r"\bkg mobility\b",
    r"\bssangyong\b",

    # China / Hong Kong
    r"\bbyd\b",
    r"\bgeely\b",
    r"\bgreat wall motor\b",
    r"\bsaic motor\b",
    r"\bshanghai automotive\b",
    r"\bnio\b",
    r"\bxpeng\b",
    r"\bli auto\b",
    r"\bleapmotor\b",
    r"\bchangan\b",
    r"\bgac group\b",
    r"\bguangzhou automobile\b",
    r"\bdongfeng motor\b",
    r"\bbaic\b",
    r"\bbeijing automotive\b",
    r"\bseres\b",
    r"\bjiangling motors\b",
    r"\bfaw\b",

    # Other likely global listings
    r"\bpolestar\b",
    r"\bvinfast\b",
    r"\btata motors\b",
    r"\bmahindra.*mahindra\b",
]

OEM_REGEX = re.compile("|".join(f"(?:{x})" for x in OEM_PATTERNS), flags=re.I)

# Optional overrides.
# Leave blank for first run.
OEM_INCLUDE_ISSUER_IDS = set()
OEM_EXCLUDE_ISSUER_IDS = set()


# ------------------------------------------------------------
# 3. FIELD DEFINITIONS
# ------------------------------------------------------------

CORE_FIELDS = [
    "issuer_id",
    "security_id",
    "issuer_name",
    "ticker",

    "standard_concept",
    "statement_type",
    "period_type",
    "expected_period_type",

    "unit_family",
    "expected_unit_family",

    "period_start",
    "period_end",

    "fiscal_year",
    "fiscal_period",

    "reported_value",
    "reported_currency",

    "filing_id",
    "document_id",

    "filing_date",
    "available_datetime",
    "available_date",

    "source_system",
]

# These are NOT automatically substituted into the data.
# They are used only to tell us whether a missing canonical field
# appears deterministically recoverable from an upstream field.

ALTERNATIVE_COLUMNS = {
    "security_id": [
        "security_id",
        "primary_security_id",
    ],

    "ticker": [
        "ticker",
        "primary_ticker",
        "listing_ticker",
        "stock_code",
        "stock_code_4d",
        "stock_code_5d",
    ],

    "fiscal_year": [
        "fiscal_year",
        "bsns_year",
        "requested_business_year",
        "business_year",
    ],

    "fiscal_period": [
        "fiscal_period",
        "requested_report_period",
        "reprt_code",
        "report_code",
        "report_type",
        "document_type_code",
        "document_description",
        "report_nm",
    ],

    "period_start": [
        "period_start",
        "periodStart",
    ],

    "period_end": [
        "period_end",
        "periodEnd",
        "reporting_date",
        "report_date",
    ],

    "available_datetime": [
        "available_datetime",
        "acceptance_datetime",
        "submit_datetime_jst",
        "release_datetime",
        "receipt_date",
        "rcept_dt",
        "processed_datetime",
    ],

    "reported_value": [
        "reported_value",
        "numeric_value",
        "value",
        "thstrm_amount",
    ],

    "reported_currency": [
        "reported_currency",
        "currency",
    ],

    "unit_family": [
        "unit_family",
        "observed_unit_family",
    ],

    "filing_id": [
        "filing_id",
        "accession_number",
        "rcept_no",
        "announcement_id",
    ],

    "document_id": [
        "document_id",
        "primary_document",
    ],
}


# ------------------------------------------------------------
# 4. HELPERS
# ------------------------------------------------------------

def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip().lower()


def nonblank_mask(series):
    if series is None:
        return pd.Series(dtype=bool)

    s = series

    mask = s.notna()

    if (
        pd.api.types.is_object_dtype(s)
        or pd.api.types.is_string_dtype(s)
    ):
        mask &= s.astype(str).str.strip().ne("")
        mask &= ~s.astype(str).str.lower().isin(
            ["nan", "none", "null", "<na>"]
        )

    return mask


def nonblank_rate(df, col):
    if col not in df.columns or len(df) == 0:
        return np.nan

    return float(nonblank_mask(df[col]).mean())


def any_candidate_rate(df, candidate_cols):
    present = [c for c in candidate_cols if c in df.columns]

    if not present or len(df) == 0:
        return np.nan

    mask = pd.Series(False, index=df.index)

    for c in present:
        mask |= nonblank_mask(df[c])

    return float(mask.mean())


def first_present(df, columns):
    return [c for c in columns if c in df.columns]


def safe_nunique(df, col):
    if col not in df.columns:
        return np.nan
    return int(df[col].dropna().nunique())


def pct(x):
    if pd.isna(x):
        return np.nan
    return round(100 * float(x), 2)


def normalise_region(x):
    s = clean_text(x)

    mapping = {
        "usa": "USA",
        "us": "USA",
        "sec": "USA",

        "europe": "EUROPE",
        "eu": "EUROPE",

        "japan": "JAPAN",

        "korea": "KOREA",
        "south korea": "KOREA",

        "mainland china": "MAINLAND_CHINA",
        "mainland_china": "MAINLAND_CHINA",
        "china": "MAINLAND_CHINA",

        "hong kong": "HONG_KONG",
        "hong_kong": "HONG_KONG",

        "australia": "AUSTRALIA",
    }

    return mapping.get(s, str(x).upper() if pd.notna(x) else None)


def determine_oem(df):
    """
    Returns boolean mask based primarily on issuer_name.
    Does NOT mutate source data.
    """

    if len(df) == 0:
        return pd.Series(False, index=df.index)

    if "issuer_name" in df.columns:
        names = df["issuer_name"].fillna("").astype(str)
    elif "entity_name" in df.columns:
        names = df["entity_name"].fillna("").astype(str)
    elif "filer_name" in df.columns:
        names = df["filer_name"].fillna("").astype(str)
    else:
        names = pd.Series("", index=df.index)

    result = names.str.contains(OEM_REGEX, na=False)

    if "issuer_id" in df.columns:

        if OEM_INCLUDE_ISSUER_IDS:
            result |= df["issuer_id"].isin(OEM_INCLUDE_ISSUER_IDS)

        if OEM_EXCLUDE_ISSUER_IDS:
            result &= ~df["issuer_id"].isin(OEM_EXCLUDE_ISSUER_IDS)

    return result


def compact_columns(df, preferred):
    return [c for c in preferred if c in df.columns]


def classify_field_problem(
    direct_rate,
    candidate_rate,
    canonical_exists,
    candidate_exists
):
    """
    Diagnostic classification only.
    """

    if canonical_exists and pd.notna(direct_rate) and direct_rate >= 0.95:
        return "OK"

    if candidate_exists:
        if pd.notna(candidate_rate) and candidate_rate >= 0.95:
            return "LIKELY_PLUMBING_OR_DETERMINISTIC_DERIVATION"

        if pd.notna(candidate_rate) and candidate_rate > 0:
            return "PARTIAL_UPSTREAM_DATA_OR_MAPPING"

    if canonical_exists:
        return "CANONICAL_FIELD_PRESENT_BUT_SPARSE"

    return "SOURCE_OR_EXTRACTION_GAP"


# ------------------------------------------------------------
# 5. LOAD FILES
# ------------------------------------------------------------

print("=" * 70)
print("LOADING EXISTING PARQUET OUTPUTS")
print("=" * 70)

data = {}
load_log = []

for name, path in FILES.items():

    exists = path.exists()

    if exists:
        try:
            df = pd.read_parquet(path)
            data[name] = df

            load_log.append({
                "dataset": name,
                "path": str(path),
                "status": "LOADED",
                "rows": len(df),
                "columns": len(df.columns),
                "file_size_mb": round(path.stat().st_size / 1_000_000, 3),
            })

            print(
                f"{name:<25} "
                f"{len(df):>10,} rows  "
                f"{len(df.columns):>4} cols"
            )

        except Exception as e:
            data[name] = pd.DataFrame()

            load_log.append({
                "dataset": name,
                "path": str(path),
                "status": f"READ_ERROR: {e}",
                "rows": 0,
                "columns": 0,
                "file_size_mb": np.nan,
            })

            print(f"ERROR reading {name}: {e}")

    else:
        data[name] = pd.DataFrame()

        load_log.append({
            "dataset": name,
            "path": str(path),
            "status": "FILE_NOT_FOUND",
            "rows": 0,
            "columns": 0,
            "file_size_mb": np.nan,
        })

        print(f"MISSING: {name} -> {path}")

load_log_df = pd.DataFrame(load_log)


# ------------------------------------------------------------
# 6. SECURITY MASTER + OEM CANDIDATES
# ------------------------------------------------------------

security_master = data["BLOCK2_SECURITY_MASTER"].copy()

if len(security_master):

    security_master["is_oem_candidate"] = determine_oem(security_master)

    candidate_cols = compact_columns(
        security_master,
        [
            "security_id",
            "issuer_id",
            "issuer_name",
            "security_name",
            "ticker",
            "listing_ticker",
            "exchange",
            "country",
            "issuer_country",
            "listing_country",
            "isin",
            "lei",
            "security_type",
            "listing_status",
        ]
    )

    oem_candidates_df = (
        security_master.loc[
            security_master["is_oem_candidate"],
            candidate_cols
        ]
        .drop_duplicates()
        .sort_values(
            [c for c in ["issuer_name", "ticker"] if c in candidate_cols]
        )
        .reset_index(drop=True)
    )

    oem_issuer_ids = set(
        security_master.loc[
            security_master["is_oem_candidate"],
            "issuer_id"
        ].dropna().astype(str)
    ) if "issuer_id" in security_master.columns else set()

    oem_security_ids = set(
        security_master.loc[
            security_master["is_oem_candidate"],
            "security_id"
        ].dropna().astype(str)
    ) if "security_id" in security_master.columns else set()

else:
    oem_candidates_df = pd.DataFrame()
    oem_issuer_ids = set()
    oem_security_ids = set()


print()
print(f"OEM candidate issuers:    {len(oem_issuer_ids):,}")
print(f"OEM candidate securities: {len(oem_security_ids):,}")


# ------------------------------------------------------------
# 7. DEFINE OEM MASK USING MASTER IDs + NAME FALLBACK
# ------------------------------------------------------------

def oem_mask(df):

    if len(df) == 0:
        return pd.Series(False, index=df.index)

    mask = pd.Series(False, index=df.index)

    if "issuer_id" in df.columns and oem_issuer_ids:
        mask |= df["issuer_id"].astype(str).isin(oem_issuer_ids)

    if "security_id" in df.columns and oem_security_ids:
        mask |= df["security_id"].astype(str).isin(oem_security_ids)

    # fallback if an upstream row does not carry IDs
    mask |= determine_oem(df)

    return mask


# ------------------------------------------------------------
# 8. DATASET INVENTORY
# ------------------------------------------------------------

inventory_rows = []

for name, df in data.items():

    if len(df):
        omask = oem_mask(df)
        oem_df = df.loc[omask]

        inventory_rows.append({
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "issuer_count": safe_nunique(df, "issuer_id"),
            "security_count": safe_nunique(df, "security_id"),
            "concept_count": safe_nunique(df, "standard_concept"),
            "oem_rows": len(oem_df),
            "oem_issuer_count": safe_nunique(oem_df, "issuer_id"),
            "oem_security_count": safe_nunique(oem_df, "security_id"),
            "oem_concept_count": safe_nunique(oem_df, "standard_concept"),
        })

    else:
        inventory_rows.append({
            "dataset": name,
            "rows": 0,
            "columns": 0,
            "issuer_count": np.nan,
            "security_count": np.nan,
            "concept_count": np.nan,
            "oem_rows": 0,
            "oem_issuer_count": np.nan,
            "oem_security_count": np.nan,
            "oem_concept_count": np.nan,
        })

dataset_inventory_df = pd.DataFrame(inventory_rows)


# ------------------------------------------------------------
# 9. SCHEMA PRESENCE
# ------------------------------------------------------------

schema_rows = []

for name, df in data.items():

    cols = set(df.columns)

    for field in CORE_FIELDS:

        candidates = ALTERNATIVE_COLUMNS.get(field, [field])
        present_candidates = [c for c in candidates if c in cols]

        schema_rows.append({
            "dataset": name,
            "canonical_field": field,
            "canonical_column_exists": field in cols,
            "alternative_columns_present": ", ".join(present_candidates),
            "any_candidate_column_exists": bool(present_candidates),
        })

schema_presence_df = pd.DataFrame(schema_rows)


# ------------------------------------------------------------
# 10. OEM COMPLETENESS BY REGIONAL INPUT
# ------------------------------------------------------------

REGIONAL_DATASETS = [
    "USA",
    "EUROPE",
    "JAPAN",
    "KOREA",
    "MAINLAND_CHINA",
    "HONG_KONG",
    "AUSTRALIA",
]

completeness_rows = []

for region in REGIONAL_DATASETS:

    df = data[region]

    if len(df) == 0:
        continue

    odf = df.loc[oem_mask(df)].copy()

    for field in CORE_FIELDS:

        canonical_exists = field in odf.columns

        direct_rate = (
            nonblank_rate(odf, field)
            if canonical_exists
            else np.nan
        )

        candidate_cols = ALTERNATIVE_COLUMNS.get(field, [field])
        present_candidate_cols = first_present(odf, candidate_cols)

        candidate_rate = any_candidate_rate(
            odf,
            candidate_cols
        )

        classification = classify_field_problem(
            direct_rate=direct_rate,
            candidate_rate=candidate_rate,
            canonical_exists=canonical_exists,
            candidate_exists=bool(present_candidate_cols),
        )

        completeness_rows.append({
            "region": region,
            "field": field,
            "oem_rows": len(odf),
            "oem_issuer_count": safe_nunique(odf, "issuer_id"),
            "canonical_column_exists": canonical_exists,
            "direct_completeness_pct": pct(direct_rate),
            "candidate_columns_present": ", ".join(present_candidate_cols),
            "candidate_completeness_pct": pct(candidate_rate),
            "diagnostic_classification": classification,
        })

regional_oem_completeness_df = pd.DataFrame(completeness_rows)


# ------------------------------------------------------------
# 11. BLOCK 10 OEM COMPLETENESS BY SOURCE REGION
# ------------------------------------------------------------

block10 = data["BLOCK10"].copy()

block10_rows = []

if len(block10):

    b10_oem = block10.loc[oem_mask(block10)].copy()

    if "source_region" in b10_oem.columns:
        b10_oem["_diagnostic_region"] = (
            b10_oem["source_region"]
            .map(normalise_region)
        )
    else:
        b10_oem["_diagnostic_region"] = "UNKNOWN"

    for region, rdf in b10_oem.groupby(
        "_diagnostic_region",
        dropna=False
    ):

        for field in CORE_FIELDS:

            rate = nonblank_rate(rdf, field)

            block10_rows.append({
                "region": region,
                "field": field,
                "oem_rows": len(rdf),
                "issuer_count": safe_nunique(rdf, "issuer_id"),
                "column_exists": field in rdf.columns,
                "completeness_pct": pct(rate),
            })

block10_oem_completeness_df = pd.DataFrame(block10_rows)


# ------------------------------------------------------------
# 12. ISSUER-LEVEL IDENTITY PROPAGATION
# ------------------------------------------------------------

identity_rows = []

for region in REGIONAL_DATASETS:

    upstream = data[region]

    if len(upstream) == 0:
        continue

    up = upstream.loc[oem_mask(upstream)].copy()

    if "issuer_id" not in up.columns:
        continue

    if len(block10) and "source_region" in block10.columns:
        b10_region = block10.loc[
            block10["source_region"]
            .map(normalise_region)
            .eq(region)
        ].copy()

        b10_region = b10_region.loc[
            oem_mask(b10_region)
        ]
    else:
        b10_region = pd.DataFrame()

    issuer_ids = sorted(
        set(up["issuer_id"].dropna().astype(str))
        |
        (
            set(b10_region["issuer_id"].dropna().astype(str))
            if "issuer_id" in b10_region.columns
            else set()
        )
    )

    for issuer in issuer_ids:

        u = up.loc[
            up["issuer_id"].astype(str).eq(issuer)
        ]

        if len(b10_region) and "issuer_id" in b10_region.columns:
            g = b10_region.loc[
                b10_region["issuer_id"].astype(str).eq(issuer)
            ]
        else:
            g = pd.DataFrame()

        issuer_name = None

        for source in [u, g]:
            if (
                len(source)
                and "issuer_name" in source.columns
                and source["issuer_name"].notna().any()
            ):
                issuer_name = (
                    source["issuer_name"]
                    .dropna()
                    .astype(str)
                    .iloc[0]
                )
                break

        up_security = nonblank_rate(u, "security_id")
        up_ticker = nonblank_rate(u, "ticker")

        b10_security = nonblank_rate(g, "security_id")
        b10_ticker = nonblank_rate(g, "ticker")

        security_loss = (
            pd.notna(up_security)
            and up_security > 0
            and (
                pd.isna(b10_security)
                or b10_security == 0
            )
        )

        ticker_loss = (
            pd.notna(up_ticker)
            and up_ticker > 0
            and (
                pd.isna(b10_ticker)
                or b10_ticker == 0
            )
        )

        identity_rows.append({
            "region": region,
            "issuer_id": issuer,
            "issuer_name": issuer_name,

            "upstream_rows": len(u),
            "block10_rows": len(g),

            "upstream_security_id_pct": pct(up_security),
            "block10_security_id_pct": pct(b10_security),

            "upstream_ticker_pct": pct(up_ticker),
            "block10_ticker_pct": pct(b10_ticker),

            "security_id_propagation_loss": security_loss,
            "ticker_propagation_loss": ticker_loss,

            "identity_issue": (
                "YES"
                if security_loss or ticker_loss
                else "NO"
            ),
        })

identity_propagation_df = pd.DataFrame(identity_rows)


# ------------------------------------------------------------
# 13. FISCAL METADATA DIAGNOSTIC BY ISSUER
# ------------------------------------------------------------

fiscal_rows = []

for region in REGIONAL_DATASETS:

    upstream = data[region]

    if len(upstream) == 0:
        continue

    up = upstream.loc[oem_mask(upstream)].copy()

    if "issuer_id" not in up.columns:
        continue

    if len(block10) and "source_region" in block10.columns:

        b10_region = block10.loc[
            block10["source_region"]
            .map(normalise_region)
            .eq(region)
        ].copy()

        b10_region = b10_region.loc[oem_mask(b10_region)]

    else:
        b10_region = pd.DataFrame()

    for issuer, u in up.groupby(
        up["issuer_id"].astype(str),
        dropna=False
    ):

        if len(b10_region) and "issuer_id" in b10_region.columns:
            g = b10_region.loc[
                b10_region["issuer_id"].astype(str).eq(str(issuer))
            ]
        else:
            g = pd.DataFrame()

        issuer_name = None

        if "issuer_name" in u.columns and u["issuer_name"].notna().any():
            issuer_name = (
                u["issuer_name"]
                .dropna()
                .astype(str)
                .iloc[0]
            )

        fy_candidates = ALTERNATIVE_COLUMNS["fiscal_year"]
        fp_candidates = ALTERNATIVE_COLUMNS["fiscal_period"]

        fiscal_rows.append({
            "region": region,
            "issuer_id": issuer,
            "issuer_name": issuer_name,

            "upstream_rows": len(u),
            "block10_rows": len(g),

            "upstream_fiscal_year_direct_pct":
                pct(nonblank_rate(u, "fiscal_year")),

            "upstream_fiscal_year_candidate_pct":
                pct(any_candidate_rate(u, fy_candidates)),

            "upstream_fiscal_year_candidate_columns":
                ", ".join(first_present(u, fy_candidates)),

            "block10_fiscal_year_pct":
                pct(nonblank_rate(g, "fiscal_year")),

            "upstream_fiscal_period_direct_pct":
                pct(nonblank_rate(u, "fiscal_period")),

            "upstream_fiscal_period_candidate_pct":
                pct(any_candidate_rate(u, fp_candidates)),

            "upstream_fiscal_period_candidate_columns":
                ", ".join(first_present(u, fp_candidates)),

            "block10_fiscal_period_pct":
                pct(nonblank_rate(g, "fiscal_period")),
        })

fiscal_diagnostics_df = pd.DataFrame(fiscal_rows)


# ------------------------------------------------------------
# 14. REGIONAL → BLOCK 10 RECONCILIATION
# ------------------------------------------------------------

reconciliation_rows = []

for region in REGIONAL_DATASETS:

    upstream = data[region]

    if len(upstream) == 0:
        continue

    up = upstream.loc[oem_mask(upstream)].copy()

    if (
        len(block10)
        and "source_region" in block10.columns
    ):

        g = block10.loc[
            block10["source_region"]
            .map(normalise_region)
            .eq(region)
        ].copy()

        g = g.loc[oem_mask(g)]

    else:
        g = pd.DataFrame()

    up_issuers = (
        set(up["issuer_id"].dropna().astype(str))
        if "issuer_id" in up.columns
        else set()
    )

    b10_issuers = (
        set(g["issuer_id"].dropna().astype(str))
        if "issuer_id" in g.columns
        else set()
    )

    up_concepts = (
        set(up["standard_concept"].dropna().astype(str))
        if "standard_concept" in up.columns
        else set()
    )

    b10_concepts = (
        set(g["standard_concept"].dropna().astype(str))
        if "standard_concept" in g.columns
        else set()
    )

    reconciliation_rows.append({
        "region": region,

        "upstream_oem_rows": len(up),
        "block10_oem_rows": len(g),

        "upstream_oem_issuers": len(up_issuers),
        "block10_oem_issuers": len(b10_issuers),

        "issuers_missing_from_block10":
            len(up_issuers - b10_issuers),

        "upstream_oem_concepts": len(up_concepts),
        "block10_oem_concepts": len(b10_concepts),

        "concepts_missing_from_block10":
            len(up_concepts - b10_concepts),

        "issuer_retention_pct":
            round(
                100 * len(up_issuers & b10_issuers) / len(up_issuers),
                2
            )
            if up_issuers else np.nan,

        "concept_retention_pct":
            round(
                100 * len(up_concepts & b10_concepts) / len(up_concepts),
                2
            )
            if up_concepts else np.nan,
    })

region_reconciliation_df = pd.DataFrame(reconciliation_rows)


# ------------------------------------------------------------
# 15. MISSING CONCEPT DETAIL
# ------------------------------------------------------------

concept_rows = []

for region in REGIONAL_DATASETS:

    up = data[region]

    if len(up) == 0:
        continue

    up = up.loc[oem_mask(up)].copy()

    if (
        len(block10)
        and "source_region" in block10.columns
    ):

        g = block10.loc[
            block10["source_region"]
            .map(normalise_region)
            .eq(region)
        ].copy()

        g = g.loc[oem_mask(g)]

    else:
        g = pd.DataFrame()

    if "standard_concept" not in up.columns:
        continue

    upstream_counts = (
        up["standard_concept"]
        .dropna()
        .astype(str)
        .value_counts()
    )

    b10_concepts = (
        set(
            g["standard_concept"]
            .dropna()
            .astype(str)
        )
        if "standard_concept" in g.columns
        else set()
    )

    for concept, count in upstream_counts.items():

        concept_rows.append({
            "region": region,
            "standard_concept": concept,
            "upstream_oem_rows": int(count),
            "appears_in_block10": concept in b10_concepts,
        })

concept_reconciliation_df = (
    pd.DataFrame(concept_rows)
    .sort_values(
        ["region", "appears_in_block10", "upstream_oem_rows"],
        ascending=[True, True, False]
    )
    if concept_rows
    else pd.DataFrame()
)


# ------------------------------------------------------------
# 16. BLOCK 9 RECONCILIATION
# ------------------------------------------------------------

block9 = data["BLOCK9"].copy()

block9_summary_rows = []

if len(block9):

    block9_oem = block9.loc[oem_mask(block9)].copy()

    block9_summary_rows.extend([
        {
            "metric": "block9_total_rows",
            "value": len(block9),
        },
        {
            "metric": "block9_oem_rows",
            "value": len(block9_oem),
        },
        {
            "metric": "block9_oem_issuers",
            "value": safe_nunique(block9_oem, "issuer_id"),
        },
    ])

    # Inspect status/decision fields rather than assuming naming.
    status_cols = [
        c for c in block9.columns
        if any(
            token in c.lower()
            for token in [
                "status",
                "accept",
                "decision",
                "publication",
                "completion",
            ]
        )
    ]

    for col in status_cols:

        vc = (
            block9_oem[col]
            .fillna("<NULL>")
            .astype(str)
            .value_counts()
            .head(25)
        )

        for value, count in vc.items():
            block9_summary_rows.append({
                "metric": f"{col} = {value}",
                "value": int(count),
            })

    # Find possible direct reconciliation keys shared with Block 10.
    candidate_join_keys = [
        "global_source_observation_id",
        "issue_signature",
        "block9_issue_signature",
        "source_row_number",
        "document_id",
        "filing_id",
    ]

    for key in candidate_join_keys:

        if (
            key in block9_oem.columns
            and key in block10.columns
        ):

            left_keys = set(
                block9_oem[key]
                .dropna()
                .astype(str)
            )

            right_keys = set(
                block10.loc[oem_mask(block10), key]
                .dropna()
                .astype(str)
            )

            matches = len(left_keys & right_keys)

            block9_summary_rows.append({
                "metric": f"JOIN_KEY_{key}_block9_unique",
                "value": len(left_keys),
            })

            block9_summary_rows.append({
                "metric": f"JOIN_KEY_{key}_block10_unique",
                "value": len(right_keys),
            })

            block9_summary_rows.append({
                "metric": f"JOIN_KEY_{key}_matches",
                "value": matches,
            })

            block9_summary_rows.append({
                "metric": f"JOIN_KEY_{key}_match_pct_of_block9",
                "value": (
                    round(100 * matches / len(left_keys), 2)
                    if left_keys else np.nan
                ),
            })

# Block 10's own Block 9 lineage fields
if len(block10):

    b10_oem = block10.loc[oem_mask(block10)].copy()

    b10_block9_cols = [
        c for c in b10_oem.columns
        if c.startswith("block9_")
    ]

    for col in b10_block9_cols:

        rate = nonblank_rate(b10_oem, col)

        block9_summary_rows.append({
            "metric": f"BLOCK10_{col}_nonblank_pct",
            "value": pct(rate),
        })

block9_reconciliation_df = pd.DataFrame(block9_summary_rows)


# ------------------------------------------------------------
# 17. COMPACT ISSUE SAMPLES
# ------------------------------------------------------------

sample_frames = []

ISSUE_FIELDS = [
    "security_id",
    "ticker",
    "fiscal_year",
    "fiscal_period",
    "available_datetime",
]

for region in REGIONAL_DATASETS:

    df = data[region]

    if len(df) == 0:
        continue

    odf = df.loc[oem_mask(df)].copy()

    if len(odf) == 0:
        continue

    for field in ISSUE_FIELDS:

        if field in odf.columns:
            bad = odf.loc[~nonblank_mask(odf[field])].copy()
        else:
            # canonical column absent = all diagnostic rows affected
            bad = odf.copy()

        if len(bad) == 0:
            continue

        sample_cols = compact_columns(
            bad,
            [
                "issuer_id",
                "security_id",
                "issuer_name",
                "ticker",
                "stock_code",
                "standard_concept",
                "filing_id",
                "document_id",
                "accession_number",
                "rcept_no",
                "period_start",
                "period_end",
                "fiscal_year",
                "fiscal_period",
                "bsns_year",
                "requested_business_year",
                "requested_report_period",
                "reprt_code",
                "report_type",
                "report_nm",
                "available_datetime",
                "filing_date",
                "source_system",
            ]
        )

        s = bad[sample_cols].head(25).copy()
        s.insert(0, "diagnostic_missing_field", field)
        s.insert(0, "diagnostic_region", region)

        sample_frames.append(s)

issue_samples_df = (
    pd.concat(sample_frames, ignore_index=True, sort=False)
    if sample_frames
    else pd.DataFrame()
)


# ------------------------------------------------------------
# 18. HIGH-PRIORITY ISSUE SUMMARY
# ------------------------------------------------------------

priority_rows = []

if len(regional_oem_completeness_df):

    for _, row in regional_oem_completeness_df.iterrows():

        classification = row["diagnostic_classification"]

        if classification == "OK":
            continue

        # Give the fields we already care most about additional priority.
        field = row["field"]

        if field in [
            "security_id",
            "ticker",
            "fiscal_year",
            "fiscal_period",
            "available_datetime",
            "period_end",
        ]:
            priority = "HIGH"
        elif field in [
            "issuer_id",
            "standard_concept",
            "reported_value",
        ]:
            priority = "CRITICAL"
        else:
            priority = "MEDIUM"

        priority_rows.append({
            "priority": priority,
            "region": row["region"],
            "field": field,
            "direct_completeness_pct":
                row["direct_completeness_pct"],
            "candidate_completeness_pct":
                row["candidate_completeness_pct"],
            "candidate_columns_present":
                row["candidate_columns_present"],
            "diagnostic_classification":
                classification,
        })

priority_order = {
    "CRITICAL": 0,
    "HIGH": 1,
    "MEDIUM": 2,
}

high_priority_issues_df = pd.DataFrame(priority_rows)

if len(high_priority_issues_df):
    high_priority_issues_df["_priority_order"] = (
        high_priority_issues_df["priority"]
        .map(priority_order)
    )

    high_priority_issues_df = (
        high_priority_issues_df
        .sort_values(
            [
                "_priority_order",
                "region",
                "field",
            ]
        )
        .drop(columns="_priority_order")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 19. README TABLE
# ------------------------------------------------------------

readme_df = pd.DataFrame(
    [
        ["Purpose",
         "Diagnose existing Blocks 1–10 for a research-grade global OEM PIT subset."],

        ["Important",
         "No source Parquet is modified by this diagnostic."],

        ["OEM classification",
         "Candidate classification based on issuer/security names plus optional ID overrides."],

        ["Interpretation: OK",
         "Canonical field is >=95% populated in the OEM regional data."],

        ["Interpretation: LIKELY_PLUMBING_OR_DETERMINISTIC_DERIVATION",
         "Canonical field is weak/absent but an upstream candidate field is >=95% populated."],

        ["Interpretation: PARTIAL_UPSTREAM_DATA_OR_MAPPING",
         "Potential upstream information exists but coverage is incomplete."],

        ["Interpretation: CANONICAL_FIELD_PRESENT_BUT_SPARSE",
         "Canonical field exists but is sparsely populated."],

        ["Interpretation: SOURCE_OR_EXTRACTION_GAP",
         "Neither canonical nor obvious deterministic candidate information is present."],

        ["Recommended next step",
         "Upload this workbook to ChatGPT. Do not rerun the production blocks first."],
    ],
    columns=["item", "description"]
)


# ------------------------------------------------------------
# 20. EXPORT TABLE COLLECTION
# ------------------------------------------------------------

tables = {
    "README": readme_df,
    "load_log": load_log_df,
    "oem_candidates": oem_candidates_df,
    "dataset_inventory": dataset_inventory_df,
    "schema_presence": schema_presence_df,
    "regional_oem_completeness": regional_oem_completeness_df,
    "block10_oem_completeness": block10_oem_completeness_df,
    "identity_propagation": identity_propagation_df,
    "fiscal_diagnostics": fiscal_diagnostics_df,
    "region_reconciliation": region_reconciliation_df,
    "concept_reconciliation": concept_reconciliation_df,
    "block9_reconciliation": block9_reconciliation_df,
    "high_priority_issues": high_priority_issues_df,
    "issue_samples": issue_samples_df,
}


# ------------------------------------------------------------
# 21. EXCEL EXPORT
# ------------------------------------------------------------

print()
print("=" * 70)
print("WRITING DIAGNOSTIC WORKBOOK")
print("=" * 70)

# Use openpyxl for this relatively small diagnostic workbook.
# This avoids the constant_memory / pandas write-order issue
# previously encountered with XlsxWriter.

try:
    import openpyxl
except ImportError:
    !pip -q install openpyxl
    import openpyxl

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    for sheet_name, df in tables.items():

        if df is None:
            continue

        # Excel maximum sheet name = 31 chars
        safe_sheet = sheet_name[:31]

        # Avoid Excel's row ceiling for safety.
        export_df = df.head(1_000_000)

        export_df.to_excel(
            writer,
            sheet_name=safe_sheet,
            index=False
        )

        ws = writer.book[safe_sheet]

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        # Modest width adjustment only.
        for column_cells in ws.columns:

            length = 0

            for cell in list(column_cells)[:250]:
                try:
                    length = max(
                        length,
                        len(str(cell.value or ""))
                    )
                except Exception:
                    pass

            ws.column_dimensions[
                column_cells[0].column_letter
            ].width = min(max(length + 2, 10), 45)


# ------------------------------------------------------------
# 22. CSV + JSON PACKAGE
# ------------------------------------------------------------

for name, df in tables.items():

    if df is None:
        continue

    path = CSV_DIR / f"{name}.csv"

    df.to_csv(
        path,
        index=False
    )

metadata = {
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "project_root":
        str(PROJECT_ROOT),

    "oem_candidate_issuer_count":
        len(oem_issuer_ids),

    "oem_candidate_security_count":
        len(oem_security_ids),

    "files": {
        name: str(path)
        for name, path in FILES.items()
    },

    "diagnostic_table_rows": {
        name: len(df)
        for name, df in tables.items()
    },
}

metadata_path = CSV_DIR / "diagnostic_metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 23. ZIP PACKAGE
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for file in sorted(CSV_DIR.glob("*")):
        zf.write(
            file,
            arcname=file.name
        )


# ------------------------------------------------------------
# 24. TERMINAL SUMMARY
# ------------------------------------------------------------

print()
print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

print(f"Workbook:")
print(EXCEL_PATH)

print()
print(f"ZIP package:")
print(ZIP_PATH)

print()
print("OEM candidate issuers:",
      f"{len(oem_issuer_ids):,}")

print("OEM candidate securities:",
      f"{len(oem_security_ids):,}")

print()

if len(high_priority_issues_df):

    print("Highest-priority detected issues:")
    display(
        high_priority_issues_df.head(40)
    )

else:
    print("No non-OK issues were identified by the basic diagnostic.")

print()
print("NEXT STEP:")
print(
    "Upload OEM_Diagnostic_Package.xlsx to ChatGPT. "
    "You do not need to upload the ZIP unless requested."
)

Mounted at /content/drive
LOADING EXISTING PARQUET OUTPUTS
BLOCK1_SECURITY_SEED           8,028 rows    15 cols
BLOCK2_SECURITY_MASTER           512 rows    34 cols
USA                          398,950 rows    67 cols
EUROPE                        13,063 rows    44 cols
JAPAN                         39,814 rows    53 cols
KOREA                         29,678 rows    71 cols
MAINLAND_CHINA                 6,985 rows    54 cols
HONG_KONG                        534 rows    61 cols
AUSTRALIA                        659 rows    47 cols
BLOCK9                        74,153 rows    71 cols
BLOCK10                      459,215 rows   102 cols

OEM candidate issuers:    47
OEM candidate securities: 55

WRITING DIAGNOSTIC WORKBOOK


ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.

In [2]:
# ============================================================
# FIX: EXPORT DIAGNOSTIC PACKAGE WITH EXCEL-SAFE DATETIMES
# ============================================================

import pandas as pd
import numpy as np
import json
import zipfile
from datetime import datetime, timezone


def make_excel_safe(df):
    """
    Return a COPY suitable for Excel export.

    Excel cannot store timezone-aware datetimes.
    Any tz-aware datetime column is converted to UTC and then
    made timezone-naive.

    Source dataframe is not modified.
    """
    out = df.copy()

    for col in out.columns:

        # Native pandas timezone-aware datetime dtype
        if isinstance(out[col].dtype, pd.DatetimeTZDtype):
            out[col] = (
                out[col]
                .dt.tz_convert("UTC")
                .dt.tz_localize(None)
            )
            continue

        # Object columns can occasionally contain Python/Pandas
        # timezone-aware datetime objects.
        if out[col].dtype == "object":

            sample = out[col].dropna().head(100)

            if len(sample) == 0:
                continue

            has_tz_datetime = sample.map(
                lambda x:
                    isinstance(x, (pd.Timestamp, datetime))
                    and getattr(x, "tzinfo", None) is not None
            ).any()

            if has_tz_datetime:
                out[col] = pd.to_datetime(
                    out[col],
                    utc=True,
                    errors="coerce"
                ).dt.tz_localize(None)

    return out


# ------------------------------------------------------------
# 1. EXCEL
# ------------------------------------------------------------

print("=" * 70)
print("WRITING EXCEL-SAFE DIAGNOSTIC WORKBOOK")
print("=" * 70)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    for sheet_name, df in tables.items():

        if df is None:
            continue

        safe_sheet = sheet_name[:31]

        # Keep the diagnostic workbook compact and under Excel limits
        export_df = df.head(1_000_000).copy()

        # Remove timezone information from export copy only
        export_df = make_excel_safe(export_df)

        export_df.to_excel(
            writer,
            sheet_name=safe_sheet,
            index=False
        )

        ws = writer.book[safe_sheet]

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        # Reasonable widths
        for column_cells in ws.columns:

            length = 0

            for cell in list(column_cells)[:250]:
                try:
                    length = max(
                        length,
                        len(str(cell.value or ""))
                    )
                except Exception:
                    pass

            ws.column_dimensions[
                column_cells[0].column_letter
            ].width = min(max(length + 2, 10), 45)


# ------------------------------------------------------------
# 2. CSV FILES
# ------------------------------------------------------------

print("Writing CSV diagnostics...")

for name, df in tables.items():

    if df is None:
        continue

    path = CSV_DIR / f"{name}.csv"

    # CSV has no Excel timezone limitation, so preserve original values
    df.to_csv(
        path,
        index=False
    )


# ------------------------------------------------------------
# 3. METADATA JSON
# ------------------------------------------------------------

metadata = {
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "project_root":
        str(PROJECT_ROOT),

    "oem_candidate_issuer_count":
        len(oem_issuer_ids),

    "oem_candidate_security_count":
        len(oem_security_ids),

    "files": {
        name: str(path)
        for name, path in FILES.items()
    },

    "diagnostic_table_rows": {
        name: len(df)
        for name, df in tables.items()
    },

    "excel_datetime_policy":
        "Timezone-aware datetimes converted to UTC and stripped of timezone for Excel export only."
}

metadata_path = CSV_DIR / "diagnostic_metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 4. ZIP CSV PACKAGE
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for file in sorted(CSV_DIR.glob("*")):
        zf.write(
            file,
            arcname=file.name
        )


# ------------------------------------------------------------
# 5. DONE
# ------------------------------------------------------------

print()
print("=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print("Workbook:")
print(EXCEL_PATH)

print()
print("ZIP:")
print(ZIP_PATH)

print()
print("OEM candidate issuers:",
      f"{len(oem_issuer_ids):,}")

print("OEM candidate securities:",
      f"{len(oem_security_ids):,}")

print()
print(
    "Upload OEM_Diagnostic_Package.xlsx to ChatGPT."
)

WRITING EXCEL-SAFE DIAGNOSTIC WORKBOOK
Writing CSV diagnostics...

EXPORT COMPLETE
Workbook:
/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/diagnostics/oem_v1/OEM_Diagnostic_Package.xlsx

ZIP:
/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/diagnostics/oem_v1/OEM_Diagnostic_Package.zip

OEM candidate issuers: 47
OEM candidate securities: 55

Upload OEM_Diagnostic_Package.xlsx to ChatGPT.
